# 01 · EDA y calidad de datos — Estadísticas de YouTube

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<usuario>/youtube-stats-nlp/blob/main/notebooks/01_eda.ipynb)

Ejecutar primero las celdas 1 y 2 (preparan el entorno igual que `00_colab_inicio`). Cambiar `SOURCE` a `'kaggle'` o `'api'` para trabajar con datos reales.

In [ ]:
# @title 1 · Clonar el repositorio e instalar dependencias
REPO_URL = "https://github.com/<usuario>/youtube-stats-nlp.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import base64, os, subprocess, sys

def _auth_args():
    """Encabezado de autenticación si existe el secreto GITHUB_TOKEN (repos privados)."""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return []
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

def _git(*args):
    r = subprocess.run(["git", *_auth_args(), *args], capture_output=True, text=True)
    if r.returncode != 0:  # no se imprime el comando: puede contener el token
        raise RuntimeError("git " + args[0] + " falló:\n" + r.stderr[-500:])

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if "<usuario>" in REPO_URL:
        raise ValueError("Cambie REPO_URL por la URL de su repositorio en GitHub")
    REPO_DIR = "/content/youtube-stats-nlp"
    if os.path.exists(REPO_DIR):
        _git("-C", REPO_DIR, "pull", "-q", "origin", BRANCH)
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_DIR}/requirements-colab.txt"], check=True)
    os.chdir(REPO_DIR)
else:  # computador local: el notebook se abre desde la carpeta notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Repositorio listo en", REPO_DIR)

In [ ]:
# @title 2 · Montar Drive y cargar secretos
USAR_DRIVE = True  # @param {type:"boolean"}
CARPETA_DRIVE = "MyDrive/ytnlp-proyecto"  # @param {type:"string"}

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from ytnlp.colab import setup
info = setup(use_drive=USAR_DRIVE, drive_folder=CARPETA_DRIVE)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ytnlp.pipeline import ingest
from ytnlp.data.validate import validate
from ytnlp.data.clean import clean_videos, clean_comments
from ytnlp.features.build_features import build

SOURCE = "sample"  # @param ["sample", "kaggle", "api"]
videos, comments = ingest(SOURCE)
videos.shape, comments.shape

## Actividad 2.1 · Diccionario de datos

In [ ]:
def diccionario(df):
    return pd.DataFrame({'tipo': df.dtypes.astype(str), '% nulos': (100*df.isna().mean()).round(2),
                         'únicos': df.nunique(), 'ejemplo': df.iloc[0]})
from IPython.display import display
display(diccionario(videos)); display(diccionario(comments))

## Actividad 3.1 · Calidad

In [ ]:
res = validate(videos, comments)
pd.json_normalize(res.report, sep=' / ').T.rename(columns={0: 'valor'})

## Distribuciones clave

In [ ]:
v, c = clean_videos(res.videos), clean_comments(res.comments)
feats = build(v, c)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
np.log10(v['views'].dropna()).hist(bins=40, ax=ax[0]); ax[0].set_title('log10(views)')
np.log10(feats['target_engagement'].dropna()).hist(bins=40, ax=ax[1]); ax[1].set_title('log10(engagement)')
c['sentiment'].value_counts().sort_index().plot.bar(ax=ax[2]); ax[2].set_title('sentimiento (0/1/2)')
plt.tight_layout()

In [ ]:
feats.groupby('keyword')['target_engagement'].median().sort_values().plot.barh(title='Engagement mediano por keyword');

## Conclusiones del equipo

- Calidad:
- Suficiencia:
- Mejoras priorizadas:
- Variable objetivo elegida: